In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, isnan, when, count

# Initialization
spark = SparkSession.builder.appName("NYC_Taxi_Stage2_3").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

# --- STAGE 1 (Fast, for linking) ---
raw_rdd = sc.textFile("../data/raw/yellow_tripdata_2016-03.csv")
no_header_rdd = raw_rdd.mapPartitionsWithIndex(lambda idx, it: it if idx > 0 else (next(it), it)[1])
split_rdd = no_header_rdd.map(lambda line: line.split(","))
clean_rdd = split_rdd.filter(lambda c: len(c) == 19)

# --- STAGE 3: DATAFRAME CONSTRUCTION (Before cleaning NaNs) ---
print("Applying manual StructType (Stage 3)...")

def convert_types(c):
    # Record repair (Stage 2 rubric): Convert empty strings '' to real Python None
    def to_float(value): return float(value) if value != '' else None
    def to_int(value): return int(value) if value != '' else None
    
    return (
        to_int(c[0]), c[1], c[2], to_int(c[3]), to_float(c[4]),
        to_float(c[5]), to_float(c[6]), to_int(c[7]), c[8],
        to_float(c[9]), to_float(c[10]), to_int(c[11]),
        to_float(c[12]), to_float(c[13]), to_float(c[14]),
        to_float(c[15]), to_float(c[16]), to_float(c[17]), to_float(c[18])
    )

typed_rdd = clean_rdd.map(convert_types)

taxi_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("RateCodeID", IntegerType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True)
])

# Promote to DataFrame with deliberate cast (Rubric stage 3, using an explicit StructType)
taxis_df = spark.createDataFrame(typed_rdd, schema=taxi_schema)

# --- STAGE 2: CLEANING AND NaNs ---
print("\n--- NULL REPORT BY COLUMN (Stage 2) ---")
# Complying with "count NaN / null per column and report them"
null_exprs = [count(when(col(c).isNull() | isnan(c), c)).alias(c) for c in taxis_df.columns]
taxis_df.select(*null_exprs).show(vertical=True)

print("\nApplying NaN policy...")
# Strategy justified by column:
# 1. Drop: If coordinates or payment fail, the trip is unusable.
no_nulls_df = taxis_df.na.drop(subset=["pickup_longitude", "dropoff_longitude", "payment_type"])
# 2. Fill: If passenger count is missing, assume the statistical mode (1 passenger).
clean_df = no_nulls_df.na.fill({"passenger_count": 1})

print("\nCalculating Outliers (IQR rule for fare_amount)...")
# Complying with "Detect and treat outliers (e.g. IQR rule)"
quantiles = clean_df.approxQuantile("fare_amount", [0.25, 0.75], 0.01)
Q1 = quantiles[0]
Q3 = quantiles[1]
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR
lower_limit = max(2.5, Q1 - 1.5 * IQR) # Never less than $2.5, which is the minimum fare

treated_df_outliers = clean_df.filter(
    (col("fare_amount") >= lower_limit) & 
    (col("fare_amount") <= upper_limit) &
    (col("trip_distance") > 0) & (col("trip_distance") < 100) &
    (col("pickup_longitude") != 0)
)

# --- FINAL EVIDENCE FOR REPORT ---
print("\n=== FINAL EVIDENCE (Stage 3: printSchema and describe) ===")
treated_df_outliers.printSchema()
treated_df_outliers.select("passenger_count", "trip_distance", "fare_amount", "tip_amount").describe().show()

# Saved to Parquet
save_path = "../data/processed/clean_taxis.parquet"
treated_df_outliers.write.mode("overwrite").parquet(save_path)
print("Clean DataFrame saved successfully!")

26/05/27 02:11:50 WARN Utils: Your hostname, LaptopPablo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/27 02:11:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 02:11:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Applying manual StructType (Stage 3)...



--- NULL REPORT BY COLUMN (Stage 2) ---



[Stage 0:=============>                                          (14 + 16) / 57]



...


[Stage 8:====================================>                   (37 + 16) / 57]




[Stage 8:=============================================>          (46 + 11) / 57]




[Stage 8:=================================================>       (49 + 8) / 57]




[Stage 8:========================================================>(56 + 1) / 57]



Clean DataFrame saved successfully!
